In [0]:
hchb_branches = dbutils.widgets.get("hchb_branches")
hchb_agencies = dbutils.widgets.get("hchb_agencies")
hchb_service_lines = dbutils.widgets.get("hchb_service_lines")
hchb_payor_types = dbutils.widgets.get("hchb_payor_types")
hchb_payor_sources = dbutils.widgets.get("hchb_payor_sources")
hchb_system_settings = dbutils.widgets.get("hchb_system_settings")
hchb_hard_close_periods = dbutils.widgets.get("hchb_hard_close_periods")
hchb_hard_close_headers = dbutils.widgets.get("hchb_hard_close_headers")
hchb_v_hard_close_header_info = dbutils.widgets.get("hchb_v_hard_close_header_info")
hchb_hard_close_revenue = dbutils.widgets.get("hchb_hard_close_revenue")
hchb_vi_hard_close_manual_adjustments = dbutils.widgets.get("hchb_vi_hard_close_manual_adjustments")
hchb_hard_close_cash = dbutils.widgets.get("hchb_hard_close_cash")
hchb_hard_close_credits = dbutils.widgets.get("hchb_hard_close_credits")
hchb_clients_all = dbutils.widgets.get("hchb_clients_all")
hchb_financial_class = dbutils.widgets.get("hchb_financial_class")
hchb_hard_close_pdgm_headers = dbutils.widgets.get("hchb_hard_close_pdgm_headers")
hchb_pdgm_period = dbutils.widgets.get("hchb_pdgm_period")
hchb_client_episode_fs = dbutils.widgets.get("hchb_client_episode_fs")
hchb_hard_close_pdgm_revenue = dbutils.widgets.get("hchb_hard_close_pdgm_revenue")
hchb_hard_close_pdgm_credits = dbutils.widgets.get("hchb_hard_close_pdgm_credits")
hchb_hard_close_pdgm_cash = dbutils.widgets.get("hchb_hard_close_pdgm_cash")
hchb_hard_close_nonpps_balance_snapshot = dbutils.widgets.get("hchb_hard_close_nonpps_balance_snapshot")
hchb_hard_close_nonpps_balance_snapshot_days = dbutils.widgets.get("hchb_hard_close_nonpps_balance_snapshot_days")
hchb_hard_close_nonpps_headers=dbutils.widgets.get("hchb_hard_close_nonpps_headers")
reportdetail_month_end_path=dbutils.widgets.get("reportdetail_month_end_path")
Report_MonthEndCloseARReport_path=dbutils.widgets.get("Report_MonthEndCloseARReport_path")
ReportDetailsPDGM_MonthEndCloseARReport_path=dbutils.widgets.get("ReportDetailsPDGM_MonthEndCloseARReport_path")
hchb_hard_close_nonpps_headers = dbutils.widgets.get("hchb_hard_close_nonpps_headers")
hchb_temp_dbo = dbutils.widgets.get("hchb_temp_dbo")

In [0]:
spark.sql(f"TRUNCATE TABLE {hchb_temp_dbo};")


In [0]:
from datetime import datetime, timedelta
from pyspark.sql import functions as F

today = datetime.now().date()

day_of_week = today.weekday()
days_to_subtract = {
    6: 0,  # Sunday
    0: 1,  # Monday
    1: 2,  # Tuesday
    2: 3,  # Wednesday
    3: 4,  # Thursday
    4: 5,  # Friday
    5: 6   # Saturday
}
reporting_week_ending_date = today - timedelta(days=days_to_subtract[day_of_week])

if today.day < 3:
    first_of_month = today.replace(day=1)
    reporting_month_end = first_of_month - timedelta(days=1)
    reporting_week_ending_month_number = reporting_month_end.month
    reporting_week_ending_year_number = reporting_month_end.year
else:
    next_month = today.replace(day=28) + timedelta(days=4)
    reporting_month_end = next_month - timedelta(days=next_month.day)
    reporting_week_ending_month_number = today.month
    reporting_week_ending_year_number = today.year

closing_period = (reporting_week_ending_year_number * 100) + reporting_week_ending_month_number
cash_received_through = (reporting_week_ending_year_number * 100) + reporting_week_ending_month_number

input_date = today if today.day >= 6 else today - timedelta(days=6)

quarter = (input_date.month - 1) // 3 + 1
if quarter == 4:
    last_day_of_quarter = input_date.replace(month=12, day=31)
else:
    next_quarter_month = quarter * 3 + 1
    last_day_of_quarter = input_date.replace(month=next_quarter_month, day=1) - timedelta(days=1)

day_of_week_last_day = last_day_of_quarter.weekday()
days_to_sunday = (day_of_week_last_day + 1) % 7
last_sunday_of_quarter = last_day_of_quarter - timedelta(days=days_to_sunday)
aging_date = last_sunday_of_quarter

ptid = None
aging_filter = None
snapshot = False
age_from_soe = False
branch_list = None
service_line_list = None
agency_list = None
exclude_lower = '-0.3'
exclude_upper = '0.3'
rptreqrpt_id = 0
psid = None
balance_filter = 0
group_by = 0
show_results = 'N'
balance_determined_at = 3

lpid = None
group_by1 = None
group_by2 = None
group_by3 = None
export_option = 1
closing_period_minus_one_period = None
period_end_date = aging_date

try:
    p_exclude_lower = float(exclude_lower)
except:
    p_exclude_lower = 0.0

try:
    p_exclude_upper = float(exclude_upper)
except:
    p_exclude_upper = 0.0

a0 = aging_filter if aging_filter is not None else 0
a1 = a0 + 30
a2 = a0 + 60
a3 = a0 + 90
a4 = a0 + 120
a5 = 180 if export_option == 1 else (a0 + 150 if export_option == 0 else 150)
a6 = 270 if export_option == 1 else (9999 if export_option == 0 else 180)
a7 = 365 if export_option == 1 else (9999 if export_option == 0 else 240)
a8 = 9999 if export_option == 0 else 365
a9 = 9999 if export_option == 0 else 531
a10 = 730 if export_option == 0 else 9999

aging_label1 = f"{a0} - {a1}"
aging_label2 = f"{a1 + 1} - {a2}"
aging_label3 = f"{a2 + 1} - {a3}"
aging_label4 = f"{a3 + 1} - {a4}"
aging_label5 = f"{a4 + 1}+" if export_option == 0 else f"{a4 + 1} - {a5}"
aging_label6 = f"{a5 + 1} - {a6}"
aging_label7 = f"{a6 + 1} - {a7}"
aging_label8 = f"{a7 + 1}+" if export_option == 1 else f"{a7 + 1} - {a8}"
aging_label9 = f"{a8 + 1} - {a9}"
aging_label10 = f"{a9 + 1} - {a10}"
aging_label11 = f"{a10 + 1}+"

pdgm_result = spark.sql(f"""
SELECT COALESCE(ss_Value, 'N') as pdgm_value
FROM {hchb_system_settings}
WHERE ss_Setting = 'PdgmShowFuturePeriodUnearned'
ORDER BY ss_value
LIMIT 1
""")

pdgm_show_future_period_unearned = 'N'
if pdgm_result.count() > 0:
    pdgm_show_future_period_unearned = pdgm_result.first()['pdgm_value']

# Get MaxPeriodDate
max_period_result = spark.sql(f"""
SELECT hcp_EndDate
FROM {hchb_hard_close_periods}
WHERE hcp_Period = {closing_period}
LIMIT 1
""")

max_period_date = None
if max_period_result.count() > 0:
    max_period_date = max_period_result.first()['hcp_EndDate']

spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW Params AS
SELECT CAST(NULL AS STRING) AS pName, CAST(NULL AS STRING) AS pValue WHERE 1=0
""")

spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW Agencies_MonthEndCloseARReport AS
SELECT CAST(NULL AS INT) AS agid, CAST(NULL AS STRING) AS Agency WHERE 1=0
""")

spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW FinancialClasses_MonthEndCloseARReport AS
SELECT CAST(NULL AS INT) AS id WHERE 1=0
""")

spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW ServiceLines_MonthEndCloseARReport AS
SELECT CAST(NULL AS INT) AS id WHERE 1=0
""")

spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW Branches_MonthEndCloseARReport AS
SELECT CAST(NULL AS STRING) AS BranchCode WHERE 1=0
""")

spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW PayorTypes_MonthEndCloseARReport AS
SELECT CAST(NULL AS INT) AS ID WHERE 1=0
""")

spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW PayorSources_MonthEndCloseARReport AS
SELECT CAST(NULL AS INT) AS id WHERE 1=0
""")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW Branches_MonthEndCloseARReport AS
SELECT Branch_Code AS BranchCode 
FROM {hchb_branches}
""")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW Agencies_MonthEndCloseARReport AS
SELECT 
    a.Agency_id AS agid,
    CONCAT(a.Agency_name, ':', a.Agency_ProviderNumber) AS Agency
FROM {hchb_agencies} AS a
""")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW ServiceLines_MonthEndCloseARReport AS
SELECT sl_id AS id 
FROM {hchb_service_lines}
""")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW PayorTypes_MonthEndCloseARReport AS
SELECT pt_id AS ID 
FROM {hchb_payor_types}
""")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW PayorSources_MonthEndCloseARReport AS
SELECT ps_id AS id 
FROM {hchb_payor_sources}
""")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW FinancialClasses_MonthEndCloseARReport AS
SELECT fc.fc_id AS id
FROM {hchb_financial_class} AS fc 
WHERE {rptreqrpt_id} = 0 
   OR EXISTS (
       SELECT 1 
       FROM {hchb_system_settings} 
       WHERE ss_setting = 'EnableFinancialClass' AND ss_value = 'N'
   )
""")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW MaxHCH_MonthEndCloseARReport AS
SELECT  
    hch_AuthID as AuthID,
    hch_msp as MSP,
    hch_fcid as FCID,
    MAX(hch_id) as HCHID
FROM {hchb_hard_close_headers}
WHERE hch_Closing_Period BETWEEN 0 AND {closing_period}
GROUP BY hch_AuthID, hch_msp, hch_fcid
""")

# MaxHCHRow_MonthEndCloseARReport
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW MaxHCHRow_MonthEndCloseARReport AS
SELECT  
    v.hch_id,
    v.hch_AuthID,
    v.hch_AuthStartDate,
    v.hch_AuthEndDate,
    v.hch_Closing_Period,
    v.hch_Closing_BranchCode,
    v.hch_InsuredId,
    v.hch_paid,
    v.hch_LastName,
    v.hch_FirstName,
    v.hch_mi,
    v.hch_ptid,
    CONCAT(v.hch_ptDesc, CASE WHEN v.hch_msp = 'TRUE' THEN '(MSP)' ELSE '' END) as hch_ptDesc,
    v.hch_msp,
    v.hch_slid,
    v.hch_slDesc,
    v.hch_fcid,
    v.hch_fcDesc,
    v.hch_psid,
    v.hch_psDesc,
    v.hch_AgencyId,
    v.hch_AgencyName,
    v.hch_ProviderNumber,
    v.hch_EOEType,
    v.hch_BillDate,
    v.hch_epiid,
    v.hch_AdmitDate,
    v.hch_DischargeDate
FROM MaxHCH_MonthEndCloseARReport mh 
JOIN {hchb_v_hard_close_header_info} AS v 
    ON v.hch_id = mh.HCHID AND v.hch_msp = mh.MSP
""")

In [0]:
if spark.catalog.tableExists(reportdetail_month_end_path):
    spark.sql(f"TRUNCATE TABLE {reportdetail_month_end_path}")
else:
    print(f"Table {reportdetail_month_end_path} does not exist, skipping truncate")

In [0]:

spark.sql(f"""
create or replace table {reportdetail_month_end_path} AS 
SELECT  
    1 as PPS, 
    h.hch_AuthID as KeyID, 
    h.hch_fcid as FCID,
    h.hch_ptid as PTID,
    h.hch_psid as psid,
    h.hch_msp as msp,
    h.hch_slid as SLID,
    h.hch_AgencyId as AgencyId,
    hcr.hcr_Reporting_BranchCode as Reporting_BranchCode,
    SUM(COALESCE(CASE WHEN hcr.hcr_type = 'E' THEN hcr.hcr_amount ELSE 0 END, 0)) as EarnedRev,
    SUM(COALESCE(CASE WHEN hcr.hcr_type = 'A' THEN hcr.hcr_amount ELSE 0 END, 0)) as Adjustments,
    SUM(COALESCE(CASE WHEN hcr.hcr_type = 'U' AND mh.AuthID IS NOT NULL THEN hcr.hcr_amount ELSE 0 END, 0)) as UnEarnedRev,
    0 as Cash,
    0 as Credits,
    0 as Refunds
FROM {hchb_hard_close_revenue} AS hcr
JOIN {hchb_hard_close_headers} AS h1 ON h1.hch_id = hcr.hcr_hchid
JOIN MaxHCHRow_MonthEndCloseARReport AS h 
    ON h.hch_AuthID = h1.hch_AuthID AND h.hch_msp = h1.hch_msp AND h.hch_fcid = h1.hch_fcid
JOIN Branches_MonthEndCloseARReport AS b ON b.BranchCode = hcr.hcr_Reporting_BranchCode
JOIN ServiceLines_MonthEndCloseARReport AS sl on sl.id = h.hch_slid
JOIN Agencies_MonthEndCloseARReport AS ag on ag.agid = h.hch_AgencyId
JOIN PayorTypes_MonthEndCloseARReport AS pt on pt.ID = h.hch_ptid
JOIN PayorSources_MonthEndCloseARReport AS ps on ps.id = h.hch_psid
JOIN FinancialClasses_MonthEndCloseARReport AS fc on fc.id = h.hch_fcid
LEFT JOIN MaxHCH_MonthEndCloseARReport AS mh on mh.HCHID = h1.hch_id
WHERE h1.hch_Closing_Period <= {closing_period}
GROUP BY h.hch_AuthID, h.hch_msp, h.hch_ptid, h.hch_psid, h.hch_slid, h.hch_AgencyId, 
         hcr.hcr_Reporting_BranchCode, h.hch_fcid
""")



In [0]:
spark.sql(f"""
insert into {reportdetail_month_end_path}
SELECT  
    1 as PPS, 
    h.hch_AuthID as KeyID, 
    h.hch_fcid as FCID,
    h.hch_ptid as PTID,
    h.hch_psid as psid,
    h.hch_msp as msp,
    h.hch_slid as SLID,
    h.hch_AgencyId as AgencyId,
    m.Reporting_BranchCode as Reporting_BranchCode,
    0 as EarnedRev,
    SUM(CASE WHEN m.reporting_period <= {closing_period}
        THEN COALESCE(m.hcma_amount, 0) ELSE 0 END) as Adjustments,
    SUM(CASE WHEN m.reporting_period > {closing_period}
        THEN COALESCE(m.hcma_amount, 0) ELSE 0 END) as UnEarnedRev,
    0 as Cash,
    0 as Credits,
    0 as Refunds
FROM {hchb_vi_hard_close_manual_adjustments} AS m
JOIN {hchb_hard_close_headers} AS h1 ON h1.hch_id = m.hch_id
JOIN MaxHCHRow_MonthEndCloseARReport AS h 
    ON h.hch_AuthID = h1.hch_AuthID AND h.hch_msp = h1.hch_msp AND h.hch_fcid = h1.hch_fcid
JOIN Branches_MonthEndCloseARReport AS b ON b.BranchCode = m.Reporting_BranchCode
JOIN ServiceLines_MonthEndCloseARReport AS sl ON sl.id = h.hch_slid
JOIN Agencies_MonthEndCloseARReport AS ag ON ag.agid = h.hch_AgencyId
JOIN PayorTypes_MonthEndCloseARReport AS pt ON pt.ID = h.hch_ptid
JOIN PayorSources_MonthEndCloseARReport AS ps ON ps.id = h.hch_psid
JOIN FinancialClasses_MonthEndCloseARReport AS fc ON fc.id = h.hch_fcid
WHERE m.hch_Closing_Period <= {closing_period}
GROUP BY h.hch_AuthID, h.hch_msp, h.hch_ptid, h.hch_psid, h.hch_slid, h.hch_AgencyId, 
         m.Reporting_BranchCode, h.hch_fcid
""")



In [0]:
spark.sql(f"""
insert into {reportdetail_month_end_path}
SELECT  
    1 as PPS, 
    h.hch_AuthID as KeyID, 
    h.hch_fcid as FCID,
    h.hch_ptid as PTID,
    h.hch_psid as psid,
    h.hch_msp as msp,
    h.hch_slid as SLID,
    h.hch_AgencyId as AgencyId,
    hcc.hcc_Reporting_BranchCode as Reporting_BranchCode,
    0 as EarnedRev,
    0 as Adjustments,
    0 as UnEarnedRev,
    SUM(COALESCE(hcc.hcc_amount, 0)) as Cash,
    0 as Credits,
    0 as Refunds
FROM {hchb_hard_close_cash} AS hcc 
JOIN {hchb_hard_close_headers} AS h1 ON h1.hch_id = hcc.hcc_hchid
JOIN MaxHCHRow_MonthEndCloseARReport AS h 
    ON h.hch_AuthID = h1.hch_AuthID AND h.hch_msp = h1.hch_msp AND h.hch_fcid = h1.hch_fcid
JOIN Branches_MonthEndCloseARReport AS b ON b.BranchCode = hcc.hcc_Reporting_BranchCode
JOIN ServiceLines_MonthEndCloseARReport AS sl ON sl.id = h.hch_slid
JOIN Agencies_MonthEndCloseARReport AS ag ON ag.agid = h.hch_AgencyId
JOIN PayorTypes_MonthEndCloseARReport AS pt ON pt.ID = h.hch_ptid
JOIN PayorSources_MonthEndCloseARReport AS ps ON ps.id = h.hch_psid
JOIN FinancialClasses_MonthEndCloseARReport AS fc ON fc.id = h.hch_fcid
WHERE ((h1.hch_Closing_Period <= {closing_period} AND {cash_received_through} IS NULL)
    OR ({cash_received_through} IS NOT NULL AND h1.hch_Closing_Period <= {cash_received_through}))
GROUP BY h.hch_AuthID, h.hch_msp, h.hch_ptid, h.hch_psid, h.hch_slid, h.hch_AgencyId, 
         hcc.hcc_Reporting_BranchCode, h.hch_fcid
""")



In [0]:
if spark.catalog.tableExists(Report_MonthEndCloseARReport_path):
    spark.sql(f"TRUNCATE TABLE {Report_MonthEndCloseARReport_path}")
else:
    print(f"Table {Report_MonthEndCloseARReport_path} does not exist, skipping truncate")

In [0]:
spark.sql(f"""
CREATE OR REPLACE table {Report_MonthEndCloseARReport_path} AS
SELECT  
    r.PPS, 
    r.KeyID, 
    m.hch_AuthStartDate as KeyDate,
    m.hch_AuthEndDate as EndDate,
    m.hch_paid as PAID,
    CONCAT(COALESCE(concat(m.hch_LastName, ', '), ''), COALESCE(m.hch_FirstName, '')) as ClientName,
    r.FCID,
    m.hch_fcDesc as Financial_Class,
    r.PTID,
    m.hch_ptDesc as Payor_Type,
    r.psid,
    m.hch_psDesc as Payor_Source,
    r.msp,
    r.SLID,
    m.hch_slDesc as slDesc,
    r.AgencyId,
    CONCAT(m.hch_AgencyName, ':', m.hch_ProviderNumber) as Agency,
    m.hch_InsuredId as InsuredId,
    m.hch_EOEType as EOEType,
    m.hch_BillDate as BillDate,
    r.Reporting_BranchCode,
    SUM(COALESCE(r.EarnedRev, 0)) as EarnedRev,
    SUM(COALESCE(r.Adjustments, 0)) as Adjustments,
    SUM(COALESCE(r.UnEarnedRev, 0)) as UnEarnedRev,
    SUM(COALESCE(r.Cash, 0)) as Cash,
    SUM(COALESCE(r.Credits, 0)) as Credits,
    SUM(COALESCE(r.Refunds, 0)) as Refunds,
    MAX(CASE WHEN datediff(m.hch_AuthEndDate, m.hch_billdate) < 0 THEN 0 
        ELSE COALESCE(datediff(m.hch_AuthEndDate, m.hch_billdate), 0) END) as DaysDelayed,
    '{period_end_date}' as LastDayClosingPeriod,
    MAX(COALESCE(CASE 
        WHEN {age_from_soe} = true THEN datediff('{period_end_date}', m.hch_authstartdate)
        ELSE datediff('{period_end_date}', m.hch_AuthEndDate)
    END, -1)) as AgingDays
FROM {reportdetail_month_end_path} as r
JOIN MaxHCHRow_MonthEndCloseARReport AS m 
    ON m.hch_AuthID = r.KeyID AND m.hch_msp = r.msp AND m.hch_fcid = r.FCID
GROUP BY 
    r.PPS, r.KeyID, m.hch_authstartdate, m.hch_AuthEndDate, m.hch_paid, CONCAT(COALESCE(concat(m.hch_LastName, ', '), ''), COALESCE(m.hch_FirstName, '')), r.FCID, m.hch_fcDesc, r.PTID, m.hch_ptDesc, 
    r.psid, m.hch_psDesc, r.msp, r.SLID, m.hch_slDesc, r.AgencyId, CONCAT(m.hch_AgencyName, ':', m.hch_ProviderNumber), m.hch_InsuredId, m.hch_EOEType, m.hch_BillDate, r.Reporting_BranchCode
""")


In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW v_hard_close_pps_credits AS
SELECT 
    hcc.hcc_closing_period,
    hcc.hcc_reporting_period,
    hcc.hcc_branchcode,
    hcc.hcc_paid,
    pa.pa_lastname,
    pa.pa_firstname,
    pa.pa_mi,
    pa.pa_MedicareNum,
    hcc.hcc_slid,
    sl.sl_desc,
    hcc.hcc_psid,
    ps.ps_desc,
    hcc.hcc_ptid,
    pt.pt_desc,
    hcc.hcc_fcid,
    fc.fc_desc,
    hcc.hcc_agencyid,
    agency.agency_name,
    agency.agency_ProviderNumber,
    hcc.hcc_hsid,
    hcc.hcc_isoffset,
    hcc.hcc_rfid,
    hcc.hcc_type,
    hcc.hcc_amount
FROM {hchb_hard_close_credits} AS hcc
LEFT JOIN {hchb_clients_all} AS pa ON pa.pa_id = hcc.hcc_paid
LEFT JOIN {hchb_payor_sources} AS ps ON ps.ps_id = hcc.hcc_psid
LEFT JOIN {hchb_payor_types} AS pt ON pt.pt_id = hcc.hcc_ptid
LEFT JOIN {hchb_financial_class} AS fc ON fc.fc_id = hcc.hcc_fcid
JOIN {hchb_agencies} AS agency ON agency.agency_id = hcc.hcc_agencyid
JOIN {hchb_service_lines} AS sl ON sl.sl_id = hcc.hcc_slid
""")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW fn_GetHardCloseCreditsAR AS
SELECT
    hcc_reporting_period AS reporting_period,
    hcc_branchcode AS closing_branchcode,
    hcc_paid AS paid,
    pa_lastname AS lastname,
    pa_firstname AS firstname,
    pa_mi AS mi,
    CONCAT(pa_lastname, ', ', pa_firstname, ' ', COALESCE(pa_mi, '')) AS fullname,
    pa_medicarenum AS medicarenum,
    hcc_slid AS slid,
    sl_desc AS sldesc,
    hcc_psid AS psid,
    ps_desc AS psdesc,
    hcc_ptid AS ptid,
    pt_desc AS ptdesc,
    hcc_fcid AS fcid,
    fc_desc AS fcdesc,
    hcc_agencyid AS agencyid,
    agency_name AS agencyname,
    agency_providernumber AS providernumber,
    hcc_type AS CreditType,
    SUM(CASE WHEN hcc_rfid IS NOT NULL THEN hcc_amount ELSE 0 END) AS CreditRefunds,
    SUM(CASE WHEN hcc_isoffset = 'True' THEN hcc_amount ELSE 0 END) AS CreditOffsets,
    SUM(CASE WHEN hcc_rfid IS NULL AND hcc_isoffset = 'False' THEN hcc_amount ELSE 0 END) AS Credits
FROM v_hard_close_pps_credits
WHERE hcc_closing_period <= {closing_period}
GROUP BY hcc_reporting_period, hcc_branchcode, hcc_paid, pa_lastname, pa_firstname, pa_mi, pa_medicarenum,
         hcc_slid, sl_desc, hcc_psid, ps_desc, hcc_ptid, pt_desc, hcc_agencyid, agency_name, agency_providernumber, hcc_type,
         hcc_fcid, fc_desc
""")



In [0]:
spark.sql(
    f"""
    INSERT INTO {Report_MonthEndCloseARReport_path}
    (
        PPS,
        KeyID,
        FCID,
        PTID,
        psid,
        msp,
        SLID,
        AgencyId,
        Reporting_Branchcode,
        EarnedRev,
        Adjustments,
        UnEarnedRev,
        Cash,
        Credits,
        Refunds,
        paid,
        financial_class,
        payor_source,
        slDesc,
        agency,
        clientname,
        InsuredId,
        EOEType,
        BillDate,
        LastDayClosingPeriod,
        keydate,
        enddate,
        payor_type,
        DaysDelayed, 
        AgingDays
    )
    SELECT  
        1 as PPS,
        0 as KeyID,
        f.fcid as FCID,
        f.ptid as PTID,
        f.psid as psid,
        0 as msp,
        f.slid as SLID,
        f.AgencyId as AgencyId,
        f.Closing_BranchCode as Reporting_BranchCode,
        0 as EarnedRev,
        0 as Adjustments,
        0 as UnEarnedRev,
        SUM(f.Credits + f.CreditRefunds + f.CreditOffsets) as Cash,
        SUM(f.Credits) as Credits,
        SUM(f.CreditRefunds * (-1)) as Refunds,
        f.paid,
        f.fcDesc as financial_class,
        f.psDesc as payor_source,
        f.slDesc as slDesc,
        concat(f.AgencyName, ':', f.providernumber) as agencyname,
        CASE 
            WHEN f.paid = 0 THEN CONCAT('CREDIT', COALESCE(' - ' || f.psDesc, ''))
            ELSE CONCAT(COALESCE(f.LastName, ''), ', ', COALESCE(f.FirstName, ''))
        END as ClientName,
        NULL as InsuredId,
        NULL as EOEType,
        NULL as BillDate,
        UNIX_DATE(CAST('{period_end_date}' AS DATE)) as LastDayClosingPeriod,
        NULL as keydate,
        NULL as enddate,
        f.ptDesc as payor_type,
        0 as DaysDelayed,
        -1 as AgingDays
    FROM fn_GetHardCloseCreditsAR as f
    JOIN Branches_MonthEndCloseARReport as b ON b.BranchCode = f.Closing_BranchCode
    JOIN ServiceLines_MonthEndCloseARReport as sl on sl.ID = f.slid
    JOIN Agencies_MonthEndCloseARReport as ag on ag.agid = f.AgencyId
    JOIN PayorTypes_MonthEndCloseARReport as pt on pt.ID = f.ptid
    JOIN PayorSources_MonthEndCloseARReport as ps on ps.ID = f.psid
    JOIN FinancialClasses_MonthEndCloseARReport as fc on fc.ID = f.fcid
    GROUP BY f.paid, f.LastName, f.FirstName, f.fcid, f.fcDesc, f.ptid, f.ptDesc, 
             f.psid, f.psDesc, f.slid, f.slDesc, f.AgencyId, f.AgencyName, f.ProviderNumber, f.Closing_BranchCode
    """
)

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW Hard_Close_Initial_PDGM_Headers AS
SELECT  MAX(hcph.hcph_id) as HCPHID,
        hcph.hcph_MSP as MSP
FROM {hchb_hard_close_pdgm_headers} AS hcph
WHERE hcph.hcph_Closing_Period <= {closing_period}
GROUP BY hcph.hcph_PDGMperiodID, hcph.hcph_MSP, hcph.hcph_FCID
""")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW Hard_Close_PDGM_Header_Details AS
SELECT  hcph.hcph_id,
        hcph.hcph_PDGMperiodID,
        hcph.hcph_PeriodStartDate,
        hcph.hcph_PeriodEndDate,
        hcph.hcph_Closing_Period,
        hcph.hcph_Closing_Branchcode,
        cefs.cefs_medicareNo as hcph_InsuredId,
        hcph.hcph_PAID,
        hcph.hcph_PTID,
        hcph.hcph_MSP,
        hcph.hcph_SLID,
        hcph.hcph_FCID,
        hcph.hcph_PSID,
        hcph.hcph_Agencyid,
        CASE WHEN hcph.hcph_IsPEP = 'True' THEN 'P'
            WHEN hcph.hcph_ReimbursementType IN('L','O') THEN hcph.hcph_ReimbursementType
            WHEN hcph.hcph_ReimbursementType = 'S' THEN NULL
            ELSE NULL END as hcph_eoetype,
        hcph.hcph_BillDate,
        hcph.hcph_epiid,
        hcph.hcph_AdmitDate,
        hcph.hcph_DischargeDate
FROM {hchb_hard_close_pdgm_headers} AS hcph 
JOIN Hard_Close_Initial_PDGM_Headers AS hcih 
    ON hcih.HCPHID = hcph.hcph_id AND CAST(hcih.MSP AS INT) = CAST(hcph.hcph_MSP AS INT)
JOIN {hchb_pdgm_period} as pp on pp.pp_id = hcph.hcph_PDGMperiodID
JOIN {hchb_client_episode_fs} as cefs on cefs.cefs_id = pp.pp_cefsid
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {ReportDetailsPDGM_MonthEndCloseARReport_path} AS
SELECT  hcph.hcph_PDGMperiodID as KeyId,
        hcph.hcph_FCID as FCID,
        hcph.hcph_PTID as PTID, 
        hcph.hcph_PSID as PSID,
        hcph.hcph_MSP as MSP, 
        hcph.hcph_SLID as SLID, 
        hcph.hcph_Agencyid as AgencyId, 
        hcpr.hcpr_Reporting_Branchcode as Reporting_Branchcode, 
        SUM(CASE WHEN hcpr.hcpr_Type = 'E' THEN hcpr.hcpr_Amount ELSE 0 END) as EarnedRev, 
        SUM(CASE WHEN hcpr.hcpr_Type = 'A' THEN hcpr.hcpr_Amount  
            WHEN hcpr.hcpr_Type = 'M' AND hcpr.hcpr_Reporting_Period <= {closing_period}
            THEN hcpr.hcpr_Amount 
            ELSE 0 END) as Adjustments,
        SUM(CASE WHEN hcpr.hcpr_Type = 'U' THEN hcpr.hcpr_Amount
            WHEN hcpr.hcpr_Type = 'M' AND hcpr.hcpr_Reporting_Period > {closing_period}
            THEN hcpr.hcpr_Amount
            ELSE 0 END) as UnEarnedRev,
        0 as Cash,
        0 as Credits,
        0 as Refunds
FROM {hchb_hard_close_pdgm_revenue} as hcpr
JOIN {hchb_hard_close_pdgm_headers} AS hcph ON hcpr.hcpr_hcphid = hcph.hcph_id
JOIN Branches_MonthEndCloseARReport AS b ON b.BranchCode = hcpr.hcpr_Reporting_Branchcode
JOIN ServiceLines_MonthEndCloseARReport AS slp ON slp.ID = hcph.hcph_slid 
JOIN Agencies_MonthEndCloseARReport AS agp ON agp.agid = hcph.hcph_Agencyid
JOIN PayorTypes_MonthEndCloseARReport AS ptp ON ptp.ID = hcph.hcph_ptid
JOIN PayorSources_MonthEndCloseARReport AS psp ON psp.ID = hcph.hcph_psid
JOIN FinancialClasses_MonthEndCloseARReport fcp ON fcp.ID = hcph.hcph_fcid
WHERE 
1 = 1 
AND hcph.hcph_Closing_Period <= {closing_period}
    AND (hcph.hcph_PeriodStartDate <= '{max_period_date}' OR '{pdgm_show_future_period_unearned}' = 'Y' )
    AND (hcpr.hcpr_Type <> 'U' OR hcph.hcph_Closing_Period = {closing_period})
GROUP BY hcph.hcph_PDGMperiodID, hcph.hcph_FCID, hcph.hcph_PTID, hcph.hcph_PSID,
        hcph.hcph_MSP, hcph.hcph_SLID, hcph.hcph_Agencyid, hcpr.hcpr_Reporting_Branchcode
""")

In [0]:
spark.sql(f"""
Insert INTO {ReportDetailsPDGM_MonthEndCloseARReport_path}
SELECT  hcph.hcph_PDGMperiodID as KeyId,
        hcph.hcph_FCID as FCID,
        hcph.hcph_PTID as PTID, 
        hcph.hcph_PSID as PSID,
        hcph.hcph_MSP as MSP, 
        hcph.hcph_SLID as SLID, 
        hcph.hcph_Agencyid as AgencyId, 
        hcpc.hcpc_Reporting_Branchcode as Reporting_Branchcode, 
        0 as EarnedRev,
        0 as Adjustments,
        0 as UnEarnedRev,
        SUM(hcpc.hcpc_Amount) as Cash,
        0 as Credits,
        0 as Refunds
FROM {hchb_hard_close_pdgm_cash} as hcpc
JOIN {hchb_hard_close_pdgm_headers} AS hcph ON hcpc.hcpc_hcphid = hcph.hcph_id
JOIN Branches_MonthEndCloseARReport AS b ON b.BranchCode = hcpc.hcpc_Reporting_Branchcode
JOIN ServiceLines_MonthEndCloseARReport AS slp ON slp.ID = hcph.hcph_slid 
JOIN Agencies_MonthEndCloseARReport AS agp ON agp.agid = hcph.hcph_Agencyid
JOIN PayorTypes_MonthEndCloseARReport AS ptp ON ptp.ID = hcph.hcph_ptid
JOIN PayorSources_MonthEndCloseARReport AS psp ON psp.ID = hcph.hcph_psid
JOIN FinancialClasses_MonthEndCloseARReport fcp ON fcp.ID = hcph.hcph_fcid
WHERE hcph.hcph_Closing_Period <= {closing_period}
    AND ({cash_received_through} IS NULL OR hcph.hcph_Closing_Period <= {cash_received_through})
    AND hcpc.hcpc_Type <> 'C'
GROUP BY hcph.hcph_PDGMperiodID, hcph.hcph_FCID, hcph.hcph_PTID, hcph.hcph_PSID,
        hcph.hcph_MSP, hcph.hcph_SLID, hcph.hcph_Agencyid, hcpc.hcpc_Reporting_Branchcode
""")

In [0]:
spark.sql(
    f"""
    INSERT INTO {Report_MonthEndCloseARReport_path}  (
        PPS,
        KeyID,
        keydate,
        enddate,
        FCID,
        PTID,
        psid,
        paid,
        msp,
        SLID,
        AgencyId,
        Reporting_Branchcode,
        clientname,
        Cash,
        Credits,
        Refunds,
        financial_class,
        payor_type,
        payor_source,
        slDesc,
        agency,
        InsuredId,
        EOEType,
        BillDate,
        DaysDelayed,
        LastDayClosingPeriod,
        AgingDays, 
        EarnedRev, 
        Adjustments, 
	    UnEarnedRev 
    )
    SELECT
        2 as PPS,
        0 as KeyId,
        null as keydate,
        NULL as enddate,
        hcpc.hcpc_FCID as FCID,
        hcpc.hcpc_PTID as PTID,
        hcpc.hcpc_psid as PSID,
        hcpc.hcpc_paid as paid,
        0 as MSP,
        hcpc.hcpc_SLID as SLID,
        hcpc.hcpc_Agencyid as AgencyId,
        hcpc.hcpc_Branchcode as Reporting_Branchcode,
        CASE
            WHEN hcpc.hcpc_paid = 0 THEN CONCAT('CREDIT', COALESCE(' - ' || ps.ps_Desc, ''))
            ELSE CONCAT(COALESCE(pa.pa_LastName, ',', pa.pa_FirstName))
        END as ClientName,
        SUM(CASE WHEN (hcpc.hcpc_rfid IS NULL AND hcpc.hcpc_isoffset = 'False')
            OR hcpc.hcpc_isoffset = 'True'
            OR hcpc.hcpc_rfid IS NOT NULL
            THEN hcpc.hcpc_amount
            ELSE 0 END) as Cash,
        SUM(CASE WHEN hcpc.hcpc_rfid IS NULL and hcpc.hcpc_isoffset = 'False'
            THEN hcpc.hcpc_amount ELSE 0 END) as Credits,
        SUM(CASE WHEN hcpc.hcpc_rfid IS NOT NULL
            THEN hcpc.hcpc_amount ELSE 0 END) as Refunds,
        fc.fc_Desc as Financial_Class,
        pt.pt_Desc as Payor_Type,
        ps.ps_Desc as Payor_Source,
        sl.sl_Desc as slDesc,
        concat(Agency.Agency_Name, ':', Agency.Agency_ProviderNumber) as Agency,
        NULL as insuredid,
        NULL as eoetype,
        NULL as billdate,
        0 as daysdelayed,
        '{period_end_date}' as lastdayclosingperiod,
        (-1) as agingdays,
        CAST(NULL AS DECIMAL(15,4)) AS EarnedRev,
        CAST(NULL AS DECIMAL(15,4)) AS Adjustments,
        CAST(NULL AS DECIMAL(15,4)) AS UnEarnedRev
    FROM {hchb_hard_close_pdgm_credits} AS hcpc
    JOIN Branches_MonthEndCloseARReport AS b ON b.BranchCode = hcpc.hcpc_BranchCode
    JOIN ServiceLines_MonthEndCloseARReport AS slp ON slp.ID = hcpc.hcpc_slid
    JOIN Agencies_MonthEndCloseARReport AS agp ON agp.agid = hcpc.hcpc_Agencyid
    JOIN PayorTypes_MonthEndCloseARReport AS ptp ON ptp.ID = hcpc.hcpc_ptid
    JOIN PayorSources_MonthEndCloseARReport AS psp ON psp.ID = hcpc.hcpc_psid
    JOIN FinancialClasses_MonthEndCloseARReport fcp ON fcp.ID = hcpc.hcpc_fcid
    LEFT JOIN {hchb_financial_class} AS fc ON fc.fc_id = hcpc.hcpc_fcid
    LEFT JOIN {hchb_clients_all} AS pa ON pa.pa_id = hcpc.hcpc_PAID
    LEFT JOIN {hchb_agencies} AS Agency ON Agency.Agency_id = hcpc.hcpc_Agencyid
    LEFT JOIN {hchb_service_lines} AS sl ON sl.sl_id = hcpc.hcpc_SLID
    LEFT JOIN {hchb_payor_types} AS pt ON pt.pt_id = hcpc.hcpc_PTID
    LEFT JOIN {hchb_payor_sources} AS ps ON ps.ps_id = hcpc.hcpc_PSID
    WHERE hcpc.hcpc_Closing_Period <= {closing_period}
    GROUP BY
        hcpc.hcpc_paid,
        hcpc.hcpc_FCID,
        hcpc.hcpc_PTID,
        hcpc.hcpc_psid,
        hcpc.hcpc_SLID,
        hcpc.hcpc_Agencyid,
       (CASE WHEN hcpc.hcpc_paid = 0 THEN CONCAT('CREDIT', COALESCE(' - ' || ps.ps_Desc, '')) ELSE CONCAT(COALESCE(pa.pa_LastName, ',', pa.pa_FirstName)) END),
        hcpc.hcpc_Branchcode,
        fc.fc_Desc,
        pt.pt_Desc,
        ps.ps_Desc,
        sl.sl_Desc,
        Agency.Agency_Name,
        concat(Agency.Agency_Name, ':', Agency.Agency_ProviderNumber),
        Agency.Agency_ProviderNumber
    """
)

In [0]:
spark.sql(f"""
INSERT INTO {Report_MonthEndCloseARReport_path}
(PPS, KeyId, KeyDate, EndDate, PAID, ClientName, FCID, Financial_Class, PTID, Payor_Type, PSID,Payor_Source, msp, SLID, SLDesc, Agencyid, Agency, 
				 InsuredId, EOEType, BillDate,Reporting_Branchcode, EarnedRev, Adjustments, UnEarnedRev, Cash, Credits, Refunds, DaysDelayed, LastDayClosingPeriod,	AgingDays)
SELECT  
    2 as PPS, 
    r.KeyId as KeyId, 
    COALESCE(hcph.hcph_PeriodStartDate, NULL) as KeyDate,
    COALESCE(hcph.hcph_PeriodEndDate, NULL) as EndDate,
    hcph.hcph_paid as PAID,
    concat(coalesce(pa.pa_LastName, ''),', ',coalesce(pa.pa_FirstName, '')) AS clientname,
    r.FCID as fcid,
    fc.fc_Desc as Financial_Class,
    r.PTID as ptid,
   CONCAT(pt.pt_Desc, CASE WHEN CAST(hcph.hcph_MSP AS INT) = 1 THEN ' (MSP)' ELSE '' END) AS Payor_Type,
    r.PSID as psid,
    ps.ps_Desc as Payor_Source,
    r.MSP as msp,
    r.SLID as slid,
    sl.sl_Desc as slDesc,
    r.AgencyId as agencyid,
    CONCAT(Agency.Agency_Name, ':', Agency.Agency_ProviderNumber) as Agency,
    hcph.hcph_InsuredId as InsuredId,
    hcph.hcph_EOEType as EOEType,
    hcph.hcph_BillDate as BillDate,
    r.Reporting_Branchcode as reporting_branchcode,
    SUM(COALESCE(r.EarnedRev, 0)) as EarnedRev,
    SUM(COALESCE(r.Adjustments, 0)) as Adjustments,
    SUM(COALESCE(r.UnEarnedRev, 0)) as UnEarnedRev,
    SUM(COALESCE(r.Cash, 0)) as Cash,
    SUM(COALESCE(r.Credits, 0)) as Credits,
    SUM(COALESCE(r.Refunds * (-1), 0)) as Refunds,
    MAX(CASE
        WHEN datediff(hcph.hcph_BillDate, hcph.hcph_PeriodEndDate) < 0 THEN 0
        ELSE coalesce(datediff(hcph.hcph_BillDate, hcph.hcph_PeriodEndDate), 0)
    END
    ) AS daysdelayed,
    '{period_end_date}' as LastDayClosingPeriod,
    MAX(CASE 
        WHEN {age_from_soe} = true THEN datediff('{period_end_date}', hcph.hcph_PeriodStartDate) 
        ELSE datediff('{period_end_date}', hcph.hcph_PeriodEndDate) 
    END) as AgingDays
FROM {ReportDetailsPDGM_MonthEndCloseARReport_path} as r
JOIN Hard_Close_PDGM_Header_Details AS hcph 
    ON hcph.hcph_PDGMperiodID = r.KeyId 
    AND CAST(hcph.hcph_MSP AS INT) = CAST(r.MSP AS INT)
    AND hcph.hcph_FCID = r.FCID
LEFT JOIN {hchb_financial_class} AS fc ON fc.fc_id = hcph.hcph_fcid 
LEFT JOIN {hchb_clients_all} AS pa ON pa.pa_id = hcph.hcph_PAID
LEFT JOIN {hchb_agencies} AS Agency ON Agency.Agency_id = hcph.hcph_Agencyid 
LEFT JOIN {hchb_service_lines} AS sl ON sl.sl_id = hcph.hcph_SLID
LEFT JOIN {hchb_payor_types} AS pt ON pt.pt_id = hcph.hcph_PTID
LEFT JOIN {hchb_payor_sources} AS ps ON ps.ps_id = hcph.hcph_PSID
GROUP BY 
    r.KeyId, hcph.hcph_PeriodStartDate, hcph.hcph_PeriodEndDate, hcph.hcph_paid,
    concat(coalesce(pa.pa_LastName, ''),', ',coalesce(pa.pa_FirstName, '')),r.FCID, fc.fc_Desc,
    r.PTID,CONCAT(pt.pt_Desc, CASE WHEN CAST(hcph.hcph_MSP AS INT) = 1 THEN ' (MSP)' ELSE '' END),r.PSID, ps.ps_Desc,r.MSP, r.SLID,
    sl.sl_Desc, r.AgencyId, CONCAT(Agency.Agency_Name, ':', Agency.Agency_ProviderNumber),
    hcph.hcph_InsuredId, hcph.hcph_EOEType, hcph.hcph_BillDate, r.Reporting_Branchcode
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW MostRecentHeader AS
SELECT  
    hcnbs_paid as paid,
    hcnbs_BranchCode as closing_branchcode,
    hcnbs_psid as psid,
    hcnbs_AgencyId as agencyid,
    hcnbs_hcnhid as hcnhid,
    ROW_NUMBER() OVER (
        PARTITION BY hcnbs_paid, hcnbs_BranchCode, hcnbs_psid, hcnbs_AgencyId 
        ORDER BY hcnbs_Closing_Period DESC
    ) as rownum
FROM {hchb_hard_close_nonpps_balance_snapshot}
WHERE hcnbs_Closing_Period <= {closing_period}
    AND hcnbs_bucketChange = False
    AND hcnbs_hcnhid IS NOT NULL
""")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW HCNHforHCNBS_MonthEndCloseARReport AS
SELECT 
    mrh.hcnhid as hcnh_id, 
    hcnbs.hcnbs_id as hcnbs_id
FROM {hchb_hard_close_nonpps_balance_snapshot} AS hcnbs
JOIN MostRecentHeader AS mrh 
    ON hcnbs.hcnbs_paid = mrh.paid
    AND hcnbs.hcnbs_BranchCode = mrh.Closing_BranchCode
    AND hcnbs.hcnbs_psid = mrh.psid
    AND hcnbs.hcnbs_AgencyId = mrh.AgencyId
WHERE hcnbs.hcnbs_Closing_Period = {closing_period}
    AND hcnbs.hcnbs_bucketChange = False
    AND mrh.rownum = 1

UNION ALL

SELECT 
    mrh.hcnhid as hcnh_id, 
    hcnbs.hcnbs_id as hcnbs_id
FROM {hchb_hard_close_nonpps_balance_snapshot} AS hcnbs
JOIN MostRecentHeader AS mrh 
    ON hcnbs.hcnbs_paid = mrh.paid
    AND hcnbs.hcnbs_BranchCode = mrh.Closing_BranchCode
    AND hcnbs.hcnbs_psid = mrh.psid
    AND hcnbs.hcnbs_AgencyId = mrh.AgencyId
WHERE hcnbs.hcnbs_Closing_Period > {closing_period}
    AND hcnbs.hcnbs_Closing_Period <= {cash_received_through}
    AND hcnbs.hcnbs_bucketChange = False
    AND mrh.rownum = 1
    AND {cash_received_through} IS NOT NULL
""")

In [0]:
spark.sql(f"""
ALTER TABLE {Report_MonthEndCloseARReport_path}
ADD COLUMNS (tcid INT)
""")


spark.sql(f"""
INSERT INTO {Report_MonthEndCloseARReport_path}
( PPS, KeyId, tcid, KeyDate, EndDate, PAID, ClientName, FCID, Financial_Class,
  PTID, Payor_Type, PSID, Payor_Source, msp, SLID, SLDesc,
  Agencyid, Agency, InsuredId, EOEType, BillDate,
  Reporting_Branchcode, EarnedRev, Adjustments, UnEarnedRev,
  Cash, Credits, Refunds, DaysDelayed, LastDayClosingPeriod, AgingDays)
    SELECT  
        0 as PPS,
        COALESCE(hcnbs.hcnbs_paid, 0) as KeyID,
        0 as tcid,
        NULL as KeyDate,
        hcnbsd.hcnbsd_dateOfService as EndDate,
        COALESCE(hcnbs.hcnbs_paid, 0) as PAID,
        CASE 
            WHEN hcnbsd.hcnbsd_DateOfService IS NULL AND hcnbs.hcnbs_paid IS NULL 
                THEN CONCAT('CREDIT', COALESCE(' - ' || ps.ps_desc, ''))
            WHEN hcnh.hcnh_id IS NOT NULL 
                THEN CAST(CONCAT(COALESCE(hcnh.hcnh_LastName, ''), CASE WHEN hcnh.hcnh_LastName IS NOT NULL THEN ', ' ELSE '' END, COALESCE(hcnh.hcnh_FirstName, '')) AS STRING)
            WHEN Clients.pa_id IS NOT NULL 
                THEN CONCAT(COALESCE(clients.pa_LastName, ''), CASE WHEN clients.pa_LastName IS NOT NULL THEN ', ' ELSE '' END, COALESCE(clients.pa_FirstName, ''))
            ELSE 'CREDIT' 
        END as ClientName,
        hcnbs.hcnbs_fcid as FCID,
        fc.fc_desc as Financial_Class,
        hcnbs.hcnbs_ptid as PTID,
        pt.pt_desc as Payor_Type,
        hcnbs.hcnbs_psid as PSID,
        ps.ps_desc as Payor_Source,
        0 as msp,
        hcnbs.hcnbs_slid as SLID,
        sl.sl_desc as slDesc,
        hcnbs.hcnbs_AgencyId as AgencyId,
        CONCAT(ag.agency_name, ':', ag.agency_ProviderNumber) as Agency,
        NULL as InsuredId,
        NULL as EOEType,
        CASE WHEN hcnbsd.hcnbsd_DateOfService IS NULL THEN NULL ELSE hcnh.hcnh_BillDate END as BillDate,
        hcnbs.hcnbs_BranchCode as Reporting_BranchCode,
        hcnbsd.hcnbsd_RevenueClosingBalance as EarnedRev,
        CAST(0 AS DECIMAL(15,4)) as Adjustments,
        CAST(0 AS DECIMAL(15,4)) as UnEarnedRev,
        hcnbsd.hcnbsd_CashClosingBalance as Cash,
        CASE WHEN hcnbsd.hcnbsd_DateOfService IS NULL THEN hcnbsd.hcnbsd_CashClosingBalance ELSE 0 END as Credits,
        CAST(0 AS DECIMAL(15,4)) as Refunds,
        0 as DaysDelayed,
        '{period_end_date}' as LastDayClosingPeriod,
        COALESCE(datediff('{period_end_date}', hcnbsd.hcnbsd_DateOfService), -1) as AgingDays
    FROM {hchb_hard_close_nonpps_balance_snapshot} AS hcnbs
    JOIN {hchb_hard_close_nonpps_balance_snapshot_days} AS hcnbsd 
        ON hcnbs.hcnbs_id = hcnbsd.hcnbsd_hcnbsid
    JOIN Branches_MonthEndCloseARReport AS bm ON bm.BranchCode = hcnbs.hcnbs_BranchCode
    JOIN ServiceLines_MonthEndCloseARReport AS slm ON slm.ID = hcnbs.hcnbs_slid
    JOIN Agencies_MonthEndCloseARReport AS agm ON agm.agid = hcnbs.hcnbs_AgencyId
    JOIN PayorTypes_MonthEndCloseARReport AS ptm ON ptm.ID = hcnbs.hcnbs_ptid
    JOIN PayorSources_MonthEndCloseARReport AS psm ON psm.ID = hcnbs.hcnbs_psid
    JOIN FinancialClasses_MonthEndCloseARReport AS fcm ON fcm.ID = hcnbs.hcnbs_fcid
    JOIN {hchb_financial_class} AS fc ON fc.fc_id = hcnbs.hcnbs_fcid
    JOIN {hchb_payor_types} AS pt ON pt.pt_id = hcnbs.hcnbs_ptid
    JOIN {hchb_payor_sources} AS ps ON ps.ps_id = hcnbs.hcnbs_psid
    JOIN {hchb_service_lines} AS sl ON sl.sl_id = hcnbs.hcnbs_slid
    JOIN {hchb_agencies} AS ag ON ag.agency_id = hcnbs.hcnbs_AgencyId
    LEFT JOIN HCNHforHCNBS_MonthEndCloseARReport AS hfh ON hcnbs.hcnbs_id = hfh.hcnbs_id
    LEFT JOIN {hchb_hard_close_nonpps_headers} AS hcnh ON hcnh.hcnh_id = hfh.hcnh_id
    LEFT JOIN {hchb_clients_all} AS clients ON clients.pa_id = hcnbs.hcnbs_paid
    WHERE hcnbs.hcnbs_Closing_Period = {closing_period}
    """)

In [0]:
spark.sql(
    f"""
    INSERT INTO {Report_MonthEndCloseARReport_path}
    (PPS, KeyID, TCID, KeyDate, EndDate, PAID, ClientName, FCID, Financial_Class, PTID, Payor_Type, psid,
     Payor_Source, SLID, slDesc, AgencyId, Agency, InsuredId, EOEType, BillDate, Reporting_BranchCode, EarnedRev, Adjustments, 
     UnEarnedRev, Cash, DaysDelayed, LastDayClosingPeriod, AgingDays, Credits, Refunds)
    SELECT  
        0 as PPS,
        COALESCE(hcnbs.hcnbs_paid, 0) as KeyID,
        0 as tcid,
        NULL as KeyDate,
        NULL as EndDate,
        COALESCE(hcnbs.hcnbs_paid, 0) as PAID,
        CASE 
            WHEN hcnbs.hcnbs_paid IS NULL THEN CONCAT('CREDIT', COALESCE(' - ' || ps.ps_desc, ''))
            WHEN hcnh.hcnh_id IS NOT NULL 
                THEN CAST(CONCAT(COALESCE(hcnh.hcnh_LastName, ''), CASE WHEN hcnh.hcnh_LastName IS NOT NULL THEN ', ' ELSE '' END, COALESCE(hcnh.hcnh_FirstName, '')) AS STRING)
            WHEN clients.pa_id IS NOT NULL 
                THEN CONCAT(COALESCE(clients.pa_LastName, ''), CASE WHEN clients.pa_LastName IS NOT NULL THEN ', ' ELSE '' END, COALESCE(clients.pa_FirstName, ''))
            ELSE 'CREDIT' 
        END as ClientName,
        hcnbs.hcnbs_fcid as FCID,
        fc.fc_desc as Financial_Class,
        hcnbs.hcnbs_ptid as PTID,
        pt.pt_desc as Payor_Type,
        hcnbs.hcnbs_psid as PSID,
        ps.ps_desc as Payor_Source,
        hcnbs.hcnbs_slid as SLID,
        sl.sl_desc as slDesc,
        hcnbs.hcnbs_AgencyId as AgencyId,
        CONCAT(ag.agency_name, ':', ag.agency_ProviderNumber) as Agency,
        NULL as InsuredId,
        NULL as EOEType,
        CASE WHEN hcnh.hcnh_BillDate IS NULL THEN NULL ELSE hcnh.hcnh_BillDate END as BillDate,
        hcnbs.hcnbs_BranchCode as Reporting_BranchCode,
        CAST(0 AS DECIMAL(15,4)) as EarnedRev,
        CAST(0 AS DECIMAL(15,4)) as Adjustments,
        hcnbs.hcnbs_unEarnedrevenue as UnEarnedRev,
        CAST(0 AS DECIMAL(15,4)) as Cash,
        0 as DaysDelayed,                                  
        CAST('{period_end_date}' AS DATE) as LastDayClosingPeriod,  
        -1 as AgingDays,                                     
        CAST(0 AS BIGINT) as Credits,                        
        CAST(0 AS DECIMAL(15,4)) as Refunds                 
    FROM {hchb_hard_close_nonpps_balance_snapshot} AS hcnbs
    JOIN Branches_MonthEndCloseARReport AS bm ON bm.BranchCode = hcnbs.hcnbs_BranchCode
    JOIN ServiceLines_MonthEndCloseARReport AS slm ON slm.ID = hcnbs.hcnbs_slid
    JOIN Agencies_MonthEndCloseARReport AS agm ON agm.agid = hcnbs.hcnbs_AgencyId
    JOIN PayorTypes_MonthEndCloseARReport AS ptm ON ptm.ID = hcnbs.hcnbs_ptid
    JOIN PayorSources_MonthEndCloseARReport AS psm ON psm.ID = hcnbs.hcnbs_psid
    JOIN FinancialClasses_MonthEndCloseARReport AS fcm ON fcm.ID = hcnbs.hcnbs_fcid
    JOIN {hchb_financial_class} AS fc ON fc.fc_id = hcnbs.hcnbs_fcid
    JOIN {hchb_payor_types} AS pt ON pt.pt_id = hcnbs.hcnbs_ptid
    JOIN {hchb_payor_sources} AS ps ON ps.ps_id = hcnbs.hcnbs_psid
    JOIN {hchb_service_lines} AS sl ON sl.sl_id = hcnbs.hcnbs_slid
    JOIN {hchb_agencies} AS ag ON ag.agency_id = hcnbs.hcnbs_AgencyId
    LEFT JOIN HCNHforHCNBS_MonthEndCloseARReport AS hfh ON hcnbs.hcnbs_id = hfh.hcnbs_id
    LEFT JOIN {hchb_hard_close_nonpps_headers} AS hcnh ON hcnh.hcnh_id = hfh.hcnh_id
    LEFT JOIN {hchb_clients_all} AS clients ON clients.pa_id = hcnbs.hcnbs_paid
    WHERE hcnbs.hcnbs_Closing_Period = {closing_period}
    AND hcnbs.hcnbs_unEarnedrevenue <> 0
    """
)

In [0]:
if cash_received_through is not None and cash_received_through > closing_period:
    
    spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW CashReceived AS
    SELECT  
        snap.hcnbs_Closing_Period, 
        snap.hcnbs_id, 
        snap.hcnbs_paid, 
        snap.hcnbs_psid, 
        snap.hcnbs_fcid,
        snap.hcnbs_ptid,
        snap.hcnbs_AgencyId, 
        snap.hcnbs_slid,
        snap.hcnbs_BranchCode, 
        lir.lir_servicedate as hcnbsd_DateOfService,
        SUM(cash.hcnc_amount) as cash, 
        SUM(CASE WHEN cash.hcnc_type = 'C' THEN cash.hcnc_amount ELSE 0 END) as credits
    FROM {hchb_hard_close_nonpps_balance_snapshot} AS snap
    JOIN {hchb_hard_close_nonpps_cash} AS cash ON cash.hcnc_hcnhid = snap.hcnbs_hcnhid
    LEFT JOIN {billing_line_items_revenue} AS lir ON lir.lir_lineitemid = cash.hcnc_tcid
    WHERE snap.hcnbs_Closing_Period > {closing_period}
        AND snap.hcnbs_Closing_Period <= {cash_received_through}
        AND snap.hcnbs_appliedCash <> 0.00
        AND EXISTS (
            SELECT 1
            FROM {accounting_revenue_details} AS detail
            JOIN {accounting_revenue_detail_closing_periods} AS rdcp 
                ON rdcp.rdcp_rdid = detail.rd_id
            JOIN {accounting_closing_periods} AS period 
                ON period.cp_id = rdcp.rdcp_cpid
            WHERE detail.rd_rtid = lir.lir_revenueid
                AND period.cp_period <= {closing_period}
        )
    GROUP BY 
        snap.hcnbs_Closing_Period, 
        snap.hcnbs_id, 
        snap.hcnbs_paid, 
        snap.hcnbs_psid, 
        snap.hcnbs_fcid, 
        snap.hcnbs_ptid, 
        snap.hcnbs_AgencyId, 
        snap.hcnbs_slid,
        snap.hcnbs_BranchCode, 
        lir.lir_servicedate
        """)
    
    spark.sql(f"""
        INSERT INTO {Report_MonthEndCloseARReport_path}
        (PPS, KeyID, TCID, KeyDate, EndDate, PAID, ClientName, FCID, Financial_Class, PTID, Payor_Type, psid, Payor_Source, SLID, 
        slDesc, AgencyId, Agency, BillDate, Reporting_BranchCode, EarnedRev, Adjustments, Cash, DaysDelayed, LastDayClosingPeriod, 
        AgingDays, Credits, Refunds)
        SELECT  
            0 as pps,
            COALESCE(hcnbs.hcnbs_paid, 0) as keyid,
            0 as tcid,
            NULL as keydate, 
            hcnbs.hcnbsd_DateOfService as enddate, 
            COALESCE(hcnbs.hcnbs_paid, 0) as paid,
            CASE 
                WHEN hcnbs.hcnbsd_DateOfService IS NULL AND hcnbs.hcnbs_paid IS NULL 
                    THEN CONCAT('CREDIT', COALESCE(' - ' || ps.ps_desc, ''))
                WHEN hcnh.hcnh_id IS NOT NULL 
                    THEN CAST(CONCAT(COALESCE(hcnh.hcnh_LastName, ''), COALESCE(', ' || hcnh.hcnh_FirstName, '')) AS STRING)
                WHEN clients.pa_id IS NOT NULL 
                    THEN CONCAT(COALESCE(clients.pa_LastName, ''), COALESCE(', ' || clients.pa_FirstName, ''))
                ELSE 'CREDIT' 
            END as clientname,  
            hcnbs.hcnbs_fcid as fcid,
            fc.fc_desc as financial_class,
            hcnbs.hcnbs_ptid as ptid,
            pt.pt_desc as payor_type,
            hcnbs.hcnbs_psid as psid,
            ps.ps_desc as payor_source,
            hcnbs.hcnbs_slid as slid,
            sl.sl_desc as sldesc,
            hcnbs.hcnbs_AgencyId as agencyid,
            CONCAT(ag.agency_name, ':', ag.agency_ProviderNumber) as agencyname,
            CASE WHEN hcnbs.hcnbsd_DateOfService IS NULL THEN NULL ELSE hcnh.hcnh_BillDate END as billdate,
            hcnbs.hcnbs_BranchCode as reporting_branchcode,
            0 as earned,
            0 as adjustments,
            hcnbs.Cash as cash,
            0 as daysdelayed,
            CAST('{period_end_date}' AS DATE) as lastdayclosingperiod,
            COALESCE(DATEDIFF(CAST('{period_end_date}' AS DATE), hcnbs.hcnbsd_DateOfService), -1) as agingdays, 
            CASE WHEN hcnbs.hcnbsd_DateOfService IS NULL THEN hcnbs.Credits ELSE 0 END as credits,
            0 as refunds
        FROM CashReceived AS hcnbs 
        JOIN Branches_MonthEndCloseARReport AS bm ON bm.BranchCode = hcnbs.hcnbs_BranchCode
        JOIN ServiceLines_MonthEndCloseARReport AS slm ON slm.ID = hcnbs.hcnbs_slid
        JOIN Agencies_MonthEndCloseARReport AS agm ON agm.agid = hcnbs.hcnbs_AgencyId
        JOIN PayorTypes_MonthEndCloseARReport AS ptm ON ptm.ID = hcnbs.hcnbs_ptid
        JOIN PayorSources_MonthEndCloseARReport AS psm ON psm.ID = hcnbs.hcnbs_psid
        JOIN FinancialClasses_MonthEndCloseARReport AS fcm ON fcm.ID = hcnbs.hcnbs_fcid
        JOIN {hchb_financial_class} AS fc ON fc.fc_id = hcnbs.hcnbs_fcid
        JOIN {hchb_payor_types} AS pt ON pt.pt_id = hcnbs.hcnbs_ptid
        JOIN {hchb_payor_sources} AS ps ON ps.ps_id = hcnbs.hcnbs_psid
        JOIN {hchb_service_lines} AS sl ON sl.sl_id = hcnbs.hcnbs_slid
        JOIN {hchb_agencies} AS ag ON ag.agency_id = hcnbs.hcnbs_AgencyId
        LEFT JOIN HCNHforHCNBS_MonthEndCloseARReport AS hfh ON hcnbs.hcnbs_id = hfh.hcnbs_id
        LEFT JOIN {hchb_hard_close_nonpps_headers} AS hcnh ON hcnh.hcnh_id = hfh.hcnh_id
        LEFT JOIN {hchb_clients_all} AS clients ON clients.pa_id = hcnbs.hcnbs_paid
    """)
else:
    print("CashReceived view NOT created (condition not met: cash_received_through <= closing_period or is None)")



In [0]:

aging_filter_sql = 'NULL' if aging_filter is None else aging_filter

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW Rts AS
SELECT 
    r.PPS,
    r.KeyID,
    MIN(r.KeyDate) as KeyDate,
    CASE 
        WHEN r.PPS = 2 THEN MAX(r.EndDate)
        WHEN r.PPS = 1 THEN MAX(r.EndDate)
        WHEN r.PPS = 0 THEN r.EndDate
    END as EndDate,
    r.PAID,
    r.ClientName,
    0 as PTID,
    '' as Payor_Type,
    r.PSID,
    r.Payor_Source,
    r.SLID,
    r.slDesc,
    r.Agency,
    r.InsuredId,
    r.EOEType,
    r.BillDate,
    r.Reporting_BranchCode,
    SUM(COALESCE(r.EarnedRev,0) + COALESCE(r.Adjustments,0) + COALESCE(r.UnEarnedRev,0)) as Revenue,
    SUM(COALESCE(r.EarnedRev,0)) as EarnedRev,
    SUM(COALESCE(r.Adjustments,0)) as Adjustments,
    SUM(COALESCE(r.UnEarnedRev,0)) as UnEarnedRev,
    SUM(COALESCE(r.Cash,0)) as Cash,
    SUM(r.Credits) as Credits,
    SUM(r.Refunds) as Refunds,
    r.AgingDays,
    COALESCE(SUM(CASE WHEN r.AgingDays < 0 
        THEN (COALESCE(r.EarnedRev,0) + COALESCE(r.Adjustments,0) - COALESCE(r.Cash,0)) 
        ELSE 0 END), 0) as A1,
    COALESCE(SUM(CASE WHEN r.AgingDays >= {a0} AND r.AgingDays <= {a1} 
        THEN (COALESCE(r.EarnedRev,0) + COALESCE(r.Adjustments,0) - COALESCE(r.Cash,0)) 
        ELSE 0 END), 0) as A2,
    COALESCE(SUM(CASE WHEN r.AgingDays > {a1} AND r.AgingDays <= {a2} 
        THEN (COALESCE(r.EarnedRev,0) + COALESCE(r.Adjustments,0) - COALESCE(r.Cash,0)) 
        ELSE 0 END), 0) as A3,
    COALESCE(SUM(CASE WHEN r.AgingDays > {a2} AND r.AgingDays <= {a3} 
        THEN (COALESCE(r.EarnedRev,0) + COALESCE(r.Adjustments,0) - COALESCE(r.Cash,0)) 
        ELSE 0 END), 0) as A4,
    COALESCE(SUM(CASE WHEN r.AgingDays > {a3} AND r.AgingDays <= {a4} 
        THEN (COALESCE(r.EarnedRev,0) + COALESCE(r.Adjustments,0) - COALESCE(r.Cash,0)) 
        ELSE 0 END), 0) as A5,
    COALESCE(SUM(CASE WHEN r.AgingDays > {a4} AND r.AgingDays <= {a5} 
        THEN (COALESCE(r.EarnedRev,0) + COALESCE(r.Adjustments,0) - COALESCE(r.Cash,0)) 
        ELSE 0 END), 0) as A6,
    COALESCE(SUM(CASE WHEN r.AgingDays > {a5} AND r.AgingDays <= {a6} 
        THEN (COALESCE(r.EarnedRev,0) + COALESCE(r.Adjustments,0) - COALESCE(r.Cash,0)) 
        ELSE 0 END), 0) as A7,
    COALESCE(SUM(CASE WHEN r.AgingDays > {a6} AND r.AgingDays <= {a7} 
        THEN (COALESCE(r.EarnedRev,0) + COALESCE(r.Adjustments,0) - COALESCE(r.Cash,0)) 
        ELSE 0 END), 0) as A8,
    COALESCE(SUM(CASE WHEN r.AgingDays > {a7} AND r.AgingDays <= {a8} 
        THEN (COALESCE(r.EarnedRev,0) + COALESCE(r.Adjustments,0) - COALESCE(r.Cash,0)) 
        ELSE 0 END), 0) as A9,
    COALESCE(SUM(CASE WHEN r.AgingDays > {a8} AND r.AgingDays <= {a9} 
        THEN (COALESCE(r.EarnedRev,0) + COALESCE(r.Adjustments,0) - COALESCE(r.Cash,0)) 
        ELSE 0 END), 0) as A10,
    COALESCE(SUM(CASE WHEN r.AgingDays > {a9} AND r.AgingDays <= {a10} 
        THEN (COALESCE(r.EarnedRev,0) + COALESCE(r.Adjustments,0) - COALESCE(r.Cash,0)) 
        ELSE 0 END), 0) as A11,
    COALESCE(SUM(CASE WHEN r.AgingDays > {a10} 
        THEN (COALESCE(r.EarnedRev,0) + COALESCE(r.Adjustments,0) - COALESCE(r.Cash,0)) 
        ELSE 0 END), 0) as A12,
    
    r.DaysDelayed,
    SUM(COALESCE(r.EarnedRev,0) + COALESCE(r.Adjustments,0) + COALESCE(r.UnEarnedRev,0) - COALESCE(r.Cash,0)) as GrossAR,
    SUM(COALESCE(r.EarnedRev,0) + COALESCE(r.Adjustments,0) - COALESCE(r.Cash,0)) as NetEarnedAR,
    COALESCE(SUM(CASE WHEN r.AgingDays < 0 
        THEN (COALESCE(r.EarnedRev,0) + COALESCE(r.Adjustments,0) + COALESCE(r.UnEarnedRev,0) - COALESCE(r.Cash,0)) 
        ELSE 0 END), 0) as EpisodeInProgress,
        0 as ExcludeDetailRow,
    '{reporting_week_ending_date}' as ReportingWeekendingDate
FROM {Report_MonthEndCloseARReport_path} r
WHERE r.AgingDays >= COALESCE({aging_filter_sql}, r.AgingDays)

GROUP BY 
    r.PPS, r.KeyID, r.PAID, r.ClientName, r.PSID, r.Payor_Source,
    r.SLID, r.slDesc, r.Agency, r.InsuredId, r.EOEType,
    r.Reporting_BranchCode, r.BillDate, r.DaysDelayed, r.AgingDays, r.EndDate
HAVING (
    ({balance_filter} = 3 AND (
        ROUND(SUM(COALESCE(r.EarnedRev,0) + COALESCE(r.Adjustments,0) + COALESCE(r.UnEarnedRev,0) - COALESCE(r.Cash,0)), 2) < {p_exclude_lower}
        OR ROUND(SUM(COALESCE(r.EarnedRev,0) + COALESCE(r.Adjustments,0) + COALESCE(r.UnEarnedRev,0) - COALESCE(r.Cash,0)), 2) > {p_exclude_upper}
        OR (ROUND(SUM(COALESCE(r.UnEarnedRev,0)), 2) < {p_exclude_lower} OR ROUND(SUM(COALESCE(r.UnEarnedRev,0)), 2) > {p_exclude_upper})
    ))
    OR ({balance_filter} = 1 AND (
        ROUND(SUM(COALESCE(r.EarnedRev,0) + COALESCE(r.Adjustments,0) + COALESCE(r.UnEarnedRev,0) - COALESCE(r.Cash,0)), 2) <= -0.01
        OR ROUND(ABS(SUM(COALESCE(r.UnEarnedRev,0))), 2) >= 0.01
    ))
    OR ({balance_filter} = 2 AND (
        ROUND(SUM(COALESCE(r.EarnedRev,0) + COALESCE(r.Adjustments,0) + COALESCE(r.UnEarnedRev,0) - COALESCE(r.Cash,0)), 2) >= 0.01
        OR ROUND(ABS(SUM(COALESCE(r.UnEarnedRev,0))), 2) >= 0.01
    ))
    OR ({balance_filter} = 0 AND (
        ROUND(ABS(SUM(COALESCE(r.EarnedRev,0) + COALESCE(r.Adjustments,0) + COALESCE(r.UnEarnedRev,0) - COALESCE(r.Cash,0))), 2) >= 0.01
        OR ROUND(ABS(SUM(COALESCE(r.UnEarnedRev,0))), 2) >= 0.01
    ))
)

ORDER BY ClientName
""")

In [0]:
spark.sql(f"""
INSERT INTO {hchb_temp_dbo}
SELECT
    'HCHB'                                  AS source_system,
    PPS                                     AS pps,
    KeyDate                                 AS key_date,
    EndDate                                 AS end_date,
    ClientName                              AS client_name,
    PAID                                    AS paid,
    Payor_Type                              AS payor_type,
    PSID                                    AS psid,
    Payor_Source                            AS payor_source,
    BillDate                                AS bill_date,
    Reporting_BranchCode                    AS reporting_branch_code,
    Revenue                                 AS revenue,
    EarnedRev                               AS earned_rev,
    UnEarnedRev                             AS unearned_rev,
    Adjustments                             AS adjustments,
    Cash                                    AS cash,
    Credits                                 AS credits,
    Refunds                                 AS refunds,
    AgingDays                               AS eoe_to_qe_days,
    CASE 
        WHEN EndDate IS NOT NULL 
            THEN datediff(CURRENT_DATE(), EndDate)
        ELSE NULL 
    END                                     AS eoe_to_me_days,
    EpisodeInProgress                       AS episode_in_progress_amt,
    A2                                      AS ar_days_0_30,
    A3                                      AS ar_days_31_60,
    A4                                      AS ar_days_61_90,
    A5                                      AS ar_days_91_120,
    (A6 + A7 + A8 + A9 + A10 + A11 + A12)    AS ar_days_121_plus,
    GrossAR                                 AS gross_ar,
    NetEarnedAR                             AS net_earned_ar,
    '{reporting_week_ending_date}'          AS reporting_week_ending_date,
    Agency                                  AS agency,
    slDesc                                  AS service_line,
    NULL                                    AS financial_class,
    InsuredId                               AS insured_id,
    EOEType                                 AS eoe_type,
    CURRENT_TIMESTAMP()                     AS created_ts,
    CURRENT_TIMESTAMP()                     AS updated_ts

FROM Rts
""")